# Support Vector Machine Classifier Implementation


---


### 01. Library Installation


In [ ]:
%pip install -qq imbalanced-learn matplotlib numpy pandas scikit-learn seaborn


### 02. Library Imports


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from imblearn.over_sampling import RandomOverSampler

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)


### 03. Data Loading and Preprocessing


Looking at the dataset that will be processed, this is the **Tic-Tac-Toe Endgame Dataset** which contains all possible board configurations at the end of tic-tac-toe games for binary classification. This dataset is used to predict whether player "X" has won or not based on the final board state.

**Dataset characteristics:**
- **Total samples**: 958 legal tic-tac-toe endgame board configurations
- **Features**: 9 categorical variables representing each square of the tic-tac-toe board
- **Target**: Binary classification (Win for X vs Loss/Draw for X)
    - Class "positive": X wins (has three-in-a-row) → mapped to 1
    - Class "negative": X loses or draws → mapped to 0

**Feature descriptions:**
Each feature represents one of the 9 squares in a tic-tac-toe board (3x3 grid):
- `top-left-square`: Top-left position
- `top-middle-square`: Top-center position
- `top-right-square`: Top-right position
- `middle-left-square`: Middle-left position
- `middle-middle-square`: Center position
- `middle-right-square`: Middle-right position
- `bottom-left-square`: Bottom-left position
- `bottom-middle-square`: Bottom-center position
- `bottom-right-square`: Bottom-right position

**Feature values** (converted to numerical):
- `x`: Player X has taken this square → mapped to 1
- `b`: Blank square (not taken) → mapped to 0
- `o`: Player O has taken this square → mapped to -1

**Dataset context:**
This dataset encodes the complete set of possible board configurations at the end of tic-tac-toe games, where "X" is assumed to have played first. The target concept is "win for X", which is true when X has one of the 8 possible ways to create a "three-in-a-row" (horizontal, vertical, or diagonal).

**Class distribution:**
- Positive cases (X wins): ~65.3% of samples (626 samples)
- Negative cases (X loses/draws): ~34.7% of samples (332 samples)

The dataset shows a moderate class imbalance favoring positive cases, which reflects the strategic advantage of playing first in tic-tac-toe.

**Data collection:**
The dataset represents all legal endgame positions where the game has concluded (either with a winner or in a draw state). This systematic enumeration ensures complete coverage of the problem space, making it an excellent dataset for machine learning algorithm evaluation and comparison.

**Research applications:**
This dataset has been widely used in machine learning research for:
- Algorithm comparison and benchmarking
- Feature construction techniques
- Rule-based learning systems
- Instance-based learning evaluation
- Decision tree algorithm testing


In [ ]:
# Importing and checking the dataframe

dataframe = pd.read_csv('../../datasets/tic_tac_toe_endgame/tic-tac-toe.data', header = None)
dataframe.head()


In [ ]:
# Renaming columns accordingly to the dataset documentation

columns_name = [
    'top-left-square',
    'top-middle-square',
    'top-right-square',
    'middle-left-square',
    'middle-middle-square',
    'middle-right-square',
    'bottom-left-square',
    'bottom-middle-square',
    'bottom-right-square',
    'Class'
]
dataframe.columns = columns_name
dataframe.head()


In [ ]:
# Checking the possible values for the target variable

dataframe['Class'].value_counts()


In [ ]:
# Changing the target variable to binary values

dataframe['Class'] = dataframe['Class'].map({'positive': 1, 'negative': 0})
dataframe['Class'].value_counts()


In [ ]:
# Changing the categorical values to numerical values
mapping = {'x': 1, 'o': -1, 'b': 0}
for column in dataframe.columns[:-1]:
    dataframe[column] = dataframe[column].map(mapping)

dataframe.head()

### 04. Data Visualization


In [ ]:
for label in dataframe:
    if label == 'Class' or dataframe[label].dtype == 'object':
        continue

    plt.figure(figsize = (10, 6))

    for class_value in dataframe['Class'].unique():
        subset = dataframe[dataframe['Class'] == class_value]
        sns.kdeplot(subset[label], label = f'Class {class_value}', fill = True, alpha = 0.5)

    plt.title(f'Distribution of {label} by Class')
    plt.xlabel(label)
    plt.ylabel('Probability')
    plt.legend(title = 'Class')
    plt.grid()    

    plt.show()


In [ ]:
# Checking the correlation matrix

plt.figure(figsize = (10, 8))
sns.heatmap(dataframe.corr(), annot = True, fmt = '.2f', cmap = 'coolwarm', square = True)
plt.title('Correlation Matrix')
plt.show()


### 05. Dataset Splitting and Scaling


In [ ]:
# Shuffling the dataset

dataframe = dataframe.sample(frac = 1, random_state = 42).reset_index(drop = True)


In [ ]:
dataframe.info()


In [ ]:
dataframe.head()


In [ ]:
# Defining the train, validation and test datasets sets

train_size = 0.7
validation_size = 0.15
test_size = 0.15


In [ ]:
# Defining the train, validation and test datasets

train_dataset = dataframe[:int(train_size * len(dataframe))]
validation_dataset = dataframe[int(train_size * len(dataframe)):int((train_size + validation_size) * len(dataframe))]
test_dataset = dataframe[int((train_size + validation_size) * len(dataframe)):]

print('\nDatasets sizes before scaling and oversampling:')
print(f'Train dataset size: {len(train_dataset)} samples')
print(f'Validation dataset size: {len(validation_dataset)} samples')
print(f'Test dataset size: {len(test_dataset)} samples')


In [ ]:
# Defining a function to scale the data and oversample if necessary

def scale_data(dataframe, oversample = False):
    features = dataframe.columns[:-1]
    target = dataframe.columns[-1]

    X = dataframe[features].values
    y = dataframe[target].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    if oversample:
        ros = RandomOverSampler(random_state = 42)
        X_resampled, y_resampled = ros.fit_resample(X_scaled, y)
        dataframe_scaled = np.hstack((X_resampled, np.reshape(y_resampled, (-1, 1))))
        return dataframe_scaled, X_resampled, y_resampled
    else:
        dataframe_scaled = np.hstack((X_scaled, np.reshape(y, (-1, 1))))
        return dataframe_scaled, X_scaled, y


In [ ]:
train_dataset, X_train, y_train = scale_data(train_dataset, oversample = True)
validation_dataset, X_validation, y_validation = scale_data(validation_dataset, oversample = False)
test_dataset, X_test, y_test = scale_data(test_dataset, oversample = False)

print('\nDatasets sizes after scaling and oversampling:')
print(f'Train dataset size: {len(train_dataset)} samples')
print(f'Validation dataset size: {len(validation_dataset)} samples')
print(f'Test dataset size: {len(test_dataset)} samples')


### 06. Support Vector Machine Classifier Implementation and Evaluation


In [ ]:
# Support Vector Machine Classifier implementation

svm_classifier_model = SVC(kernel = 'rbf', C = 1.0, gamma = 'scale', probability = True, random_state = 42)
svm_classifier_model.fit(X_train, y_train)

y_predictions_validation = svm_classifier_model.predict(X_validation)
y_predictions_test = svm_classifier_model.predict(X_test)


In [ ]:
# Classification reports

print('Validation Set Classification Report:')
print(classification_report(y_validation, y_predictions_validation))

print('Test Set Classification Report:')
print(classification_report(y_test, y_predictions_test))


In [ ]:
# Confusion matrix

confusion_matrix_validation = confusion_matrix(y_validation, y_predictions_validation)
confusion_matrix_test = confusion_matrix(y_test, y_predictions_test)

plt.figure(figsize = (8, 6))
sns.heatmap(confusion_matrix_validation, annot = True, fmt = 'd', cmap = 'Blues', cbar = False)
plt.title('Confusion Matrix - Validation Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

plt.figure(figsize = (8, 6))
sns.heatmap(confusion_matrix_test, annot = True, fmt = 'd', cmap = 'Greens', cbar = False)
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


In [ ]:
# Getting probability predictions for additional metrics

y_pred_proba_validation = svm_classifier_model.predict_proba(X_validation)
y_pred_proba_test = svm_classifier_model.predict_proba(X_test)


In [ ]:
# Basic accuracy metrics

accuracy_validation = accuracy_score(y_validation, y_predictions_validation)
accuracy_test = accuracy_score(y_test, y_predictions_test)

print(f'Validation Set Accuracy: {accuracy_validation:.4f}')
print(f'Test Set Accuracy: {accuracy_test:.4f}')


In [ ]:
# Precision metrics

precision_validation = precision_score(y_validation, y_predictions_validation, average = 'weighted')
precision_test = precision_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set Precision (weighted): {precision_validation:.4f}')
print(f'Test Set Precision (weighted): {precision_test:.4f}')

# Class-specific precision
precision_per_class_validation = precision_score(y_validation, y_predictions_validation, average = None)
precision_per_class_test = precision_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific Precision:')
print(f'\tNegative (0): {precision_per_class_validation[0]:.4f}')
print(f'\tPositive (1): {precision_per_class_validation[1]:.4f}')

print(f'\nTest Set - Class-specific Precision:')
print(f'\tNegative (0): {precision_per_class_test[0]:.4f}')
print(f'\tPositive (1): {precision_per_class_test[1]:.4f}')


In [ ]:
# Recall metrics

recall_validation = recall_score(y_validation, y_predictions_validation, average = 'weighted')
recall_test = recall_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set Recall (weighted): {recall_validation:.4f}')
print(f'Test Set Recall (weighted): {recall_test:.4f}')

# Class-specific recall
recall_per_class_validation = recall_score(y_validation, y_predictions_validation, average = None)
recall_per_class_test = recall_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific Recall:')
print(f'\tNegative (0): {recall_per_class_validation[0]:.4f}')
print(f'\tPositive (1): {recall_per_class_validation[1]:.4f}')

print(f'\nTest Set - Class-specific Recall:')
print(f'\tNegative (0): {recall_per_class_test[0]:.4f}')
print(f'\tPositive (1): {recall_per_class_test[1]:.4f}')


In [ ]:
# F1-Score metrics

f1_validation = f1_score(y_validation, y_predictions_validation, average = 'weighted')
f1_test = f1_score(y_test, y_predictions_test, average = 'weighted')

print(f'Validation Set F1-Score (weighted): {f1_validation:.4f}')
print(f'Test Set F1-Score (weighted): {f1_test:.4f}')

# Class-specific F1-Score
f1_per_class_validation = f1_score(y_validation, y_predictions_validation, average = None)
f1_per_class_test = f1_score(y_test, y_predictions_test, average = None)

print(f'\nValidation Set - Class-specific F1-Score:')
print(f'\tNegative (0): {f1_per_class_validation[0]:.4f}')
print(f'\tPositive (1): {f1_per_class_validation[1]:.4f}')

print(f'\nTest Set - Class-specific F1-Score:')
print(f'\tNegative (0): {f1_per_class_test[0]:.4f}')
print(f'\tPositive (1): {f1_per_class_test[1]:.4f}')


In [ ]:
# Balanced accuracy

balanced_acc_validation = balanced_accuracy_score(y_validation, y_predictions_validation)
balanced_acc_test = balanced_accuracy_score(y_test, y_predictions_test)

print(f'Validation Set Balanced Accuracy: {balanced_acc_validation:.4f}')
print(f'Test Set Balanced Accuracy: {balanced_acc_test:.4f}')


In [ ]:
# Matthews Correlation Coefficient (MCC)

mcc_validation = matthews_corrcoef(y_validation, y_predictions_validation)
mcc_test = matthews_corrcoef(y_test, y_predictions_test)

print(f'Validation Set Matthews Correlation Coefficient: {mcc_validation:.4f}')
print(f'Test Set Matthews Correlation Coefficient: {mcc_test:.4f}')


In [ ]:
# Cohen's Kappa

kappa_validation = cohen_kappa_score(y_validation, y_predictions_validation)
kappa_test = cohen_kappa_score(y_test, y_predictions_test)

print(f'Validation Set Cohen\'s Kappa: {kappa_validation:.4f}')
print(f'Test Set Cohen\'s Kappa: {kappa_test:.4f}')


In [ ]:
# ROC AUC

roc_auc_validation = roc_auc_score(y_validation, y_pred_proba_validation[:, 1])
roc_auc_test = roc_auc_score(y_test, y_pred_proba_test[:, 1])

print(f'Validation Set ROC AUC: {roc_auc_validation:.4f}')
print(f'Test Set ROC AUC: {roc_auc_test:.4f}')


In [ ]:
# Average Precision (PR AUC)

avg_precision_validation = average_precision_score(y_validation, y_pred_proba_validation[:, 1])
avg_precision_test = average_precision_score(y_test, y_pred_proba_test[:, 1])

print(f'Validation Set Average Precision (PR AUC): {avg_precision_validation:.4f}')
print(f'Test Set Average Precision (PR AUC): {avg_precision_test:.4f}')


In [ ]:
# Log Loss

logloss_validation = log_loss(y_validation, y_pred_proba_validation)
logloss_test = log_loss(y_test, y_pred_proba_test)

print(f'Validation Set Log Loss: {logloss_validation:.4f}')
print(f'Test Set Log Loss: {logloss_test:.4f}')


In [ ]:
# ROC Curve - Validation Set

fpr_val, tpr_val, _ = roc_curve(y_validation, y_pred_proba_validation[:, 1])

plt.figure(figsize = (8, 6))
plt.plot(fpr_val, tpr_val, color = 'darkorange', lw = 2, label = f'ROC curve (AUC = {roc_auc_validation:.4f})')
plt.plot([0, 1], [0, 1], color = 'navy', lw = 2, linestyle = '--', label = 'Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Validation Set')
plt.legend(loc = "lower right")
plt.grid(True)
plt.show()


In [ ]:
# ROC Curve - Test Set

fpr_test, tpr_test, _ = roc_curve(y_test, y_pred_proba_test[:, 1])

plt.figure(figsize = (8, 6))
plt.plot(fpr_test, tpr_test, color = 'darkorange', lw = 2, label = f'ROC curve (AUC = {roc_auc_test:.4f})')
plt.plot([0, 1], [0, 1], color = 'navy', lw = 2, linestyle = '--', label = 'Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Test Set')
plt.legend(loc = "lower right")
plt.grid(True)
plt.show()


In [ ]:
# Precision-Recall Curve - Validation Set

precision_val, recall_val, _ = precision_recall_curve(y_validation, y_pred_proba_validation[:, 1])

plt.figure(figsize = (8, 6))
plt.plot(recall_val, precision_val, color = 'blue', lw = 2, label = f'PR curve (AP = {avg_precision_validation:.4f})')
plt.axhline(y = np.mean(y_validation), color = 'red', linestyle = '--', label = f'Random Classifier (AP = {np.mean(y_validation):.4f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Validation Set')
plt.legend(loc = "lower left")
plt.grid(True)
plt.show()


In [ ]:
# Precision-Recall Curve - Test Set

precision_test, recall_test, _ = precision_recall_curve(y_test, y_pred_proba_test[:, 1])

plt.figure(figsize = (8, 6))
plt.plot(recall_test, precision_test, color = 'blue', lw = 2, label = f'PR curve (AP = {avg_precision_test:.4f})')
plt.axhline(y = np.mean(y_test), color = 'red', linestyle = '--', label = f'Random Classifier (AP = {np.mean(y_test):.4f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Test Set')
plt.legend(loc = "lower left")
plt.grid(True)
plt.show()


In [ ]:
# Cross-validation with accuracy

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
cv_accuracy_scores = cross_val_score(svm_classifier_model, X_train, y_train, cv = cv, scoring = 'accuracy')

print(f'Cross-validation Accuracy:')
print(f'\tMean: {cv_accuracy_scores.mean():.4f} (+/- {cv_accuracy_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_accuracy_scores]}')


In [ ]:
# Cross-validation with precision

cv_precision_scores = cross_val_score(svm_classifier_model, X_train, y_train, cv = cv, scoring = 'precision_weighted')

print(f'Cross-validation Precision (weighted):')
print(f'\tMean: {cv_precision_scores.mean():.4f} (+/- {cv_precision_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_precision_scores]}')


In [ ]:
# Cross-validation with recall

cv_recall_scores = cross_val_score(svm_classifier_model, X_train, y_train, cv = cv, scoring = 'recall_weighted')

print(f'Cross-validation Recall (weighted):')
print(f'\tMean: {cv_recall_scores.mean():.4f} (+/- {cv_recall_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_recall_scores]}')


In [ ]:
# Cross-validation with F1-Score

cv_f1_scores = cross_val_score(svm_classifier_model, X_train, y_train, cv = cv, scoring = 'f1_weighted')

print(f'Cross-validation F1-Score (weighted):')
print(f'\tMean: {cv_f1_scores.mean():.4f} (+/- {cv_f1_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_f1_scores]}')


In [ ]:
# Cross-validation with ROC AUC

cv_roc_auc_scores = cross_val_score(svm_classifier_model, X_train, y_train, cv = cv, scoring = 'roc_auc')

print(f'Cross-validation ROC AUC:')
print(f'\tMean: {cv_roc_auc_scores.mean():.4f} (+/- {cv_roc_auc_scores.std() * 2:.4f})')
print(f'\tIndividual folds: {[f"{score:.4f}" for score in cv_roc_auc_scores]}')


In [ ]:
# Error analysis - Validation Set

misclassified_mask_validation = y_validation != y_predictions_validation
misclassified_indices_validation = np.where(misclassified_mask_validation)[0]

print(f'Validation Set Error Analysis:')
print(f'Total misclassified samples: {len(misclassified_indices_validation)} out of {len(y_validation)}')
print(f'Error rate: {len(misclassified_indices_validation)/len(y_validation)*100:.2f}%')

# Analyze misclassification by true class
for true_class in [0, 1]:
    class_name = 'Negative' if true_class == 0 else 'Positive'
    true_class_mask = y_validation == true_class
    misclass_in_class = np.sum(misclassified_mask_validation & true_class_mask)
    total_in_class = np.sum(true_class_mask)

    print(f'{class_name} cases (Class {true_class}):')
    print(f'\tMisclassified: {misclass_in_class}/{total_in_class} ({misclass_in_class/total_in_class*100:.2f}%)')


In [ ]:
# Error analysis - Test Set

misclassified_mask_test = y_test != y_predictions_test
misclassified_indices_test = np.where(misclassified_mask_test)[0]

print(f'Test Set Error Analysis:')
print(f'Total misclassified samples: {len(misclassified_indices_test)} out of {len(y_test)}')
print(f'Error rate: {len(misclassified_indices_test)/len(y_test)*100:.2f}%')

# Analyze misclassification by true class
for true_class in [0, 1]:
    class_name = 'Negative' if true_class == 0 else 'Positive'
    true_class_mask = y_test == true_class
    misclass_in_class = np.sum(misclassified_mask_test & true_class_mask)
    total_in_class = np.sum(true_class_mask)

    print(f'{class_name} cases (Class {true_class}):')
    print(f'\tMisclassified: {misclass_in_class}/{total_in_class} ({misclass_in_class/total_in_class*100:.2f}%)')


In [ ]:
# Confidence analysis

# Misclassified samples confidence
misclass_confidences_validation = np.max(y_pred_proba_validation[misclassified_mask_validation], axis = 1)
correct_confidences_validation = np.max(y_pred_proba_validation[misclassified_mask_validation], axis = 1)

misclass_confidences_test = np.max(y_pred_proba_test[misclassified_mask_test], axis = 1)
correct_confidences_test = np.max(y_pred_proba_test[misclassified_mask_test], axis = 1)

print('Confidence Analysis:')
print(f'Validation Set:')
print(f'\tAverage confidence of misclassified samples: {misclass_confidences_validation.mean():.4f}')
print(f'\tAverage confidence of correct predictions: {correct_confidences_validation.mean():.4f}')

print(f'Test Set:')
print(f'\tAverage confidence of misclassified samples: {misclass_confidences_test.mean():.4f}')
print(f'\tAverage confidence of correct predictions: {correct_confidences_test.mean():.4f}')


In [ ]:
# Learning curve analysis

train_sizes = np.linspace(0.1, 1.0, 10)

train_sizes_abs, train_scores, val_scores = learning_curve(
    svm_classifier_model, X_train, y_train, train_sizes = train_sizes, cv = 5, 
    scoring = 'accuracy', random_state = 42, n_jobs = -1,
)

train_mean = np.mean(train_scores, axis = 1)
train_std = np.std(train_scores, axis = 1)
val_mean = np.mean(val_scores, axis = 1)
val_std = np.std(val_scores, axis = 1)

plt.figure(figsize = (10, 6))
plt.plot(train_sizes_abs, train_mean, 'o-', color = 'blue', label = 'Training Accuracy')
plt.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha = 0.1, color = 'blue')

plt.plot(train_sizes_abs, val_mean, 'o-', color = 'red', label = 'Validation Accuracy')
plt.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha = 0.1, color = 'red')

plt.xlabel('Training Set Size')
plt.ylabel('Accuracy Score')
plt.title('Learning Curve - Support Vector Machine Classifier')
plt.legend()
plt.grid(True)
plt.show()


### 07. Summary and Interpretation of Additional Metrics

The additional metrics provide deeper insights into your Support Vector Machine model performance:

**Basic Classification Metrics:**
- **Balanced Accuracy**: Accounts for class imbalance better than regular accuracy
- **Matthews Correlation Coefficient (MCC)**: Measures correlation between observed and predicted classifications (-1 to +1, where +1 is perfect)
- **Cohen's Kappa**: Inter-rater reliability statistic that accounts for agreement by chance
- **Class-specific metrics**: Individual precision, recall, and F1 for each class

**Probability-based Metrics:**
- **ROC-AUC**: Area under the ROC curve, measures discriminative ability
- **Average Precision (PR-AUC)**: Area under Precision-Recall curve, better for imbalanced datasets
- **Log Loss**: Quantifies the uncertainty of predictions based on probabilities

**Analysis Techniques:**
- **Cross-validation**: Provides robust estimates with confidence intervals
- **Error Analysis**: Identifies patterns in misclassified samples and confidence levels
- **Learning Curve**: Shows if more data would improve performance

**ROC and PR Curves:**
- ROC curves show trade-off between sensitivity and specificity
- PR curves are more informative for imbalanced datasets
- Higher AUC values indicate better performance

**Support Vector Machine Specific Insights:**
- **Kernel Performance**: RBF kernel effectively handles non-linear decision boundaries in the tic-tac-toe feature space
- **Hyperparameter Sensitivity**: The model performance is influenced by C (regularization) and gamma (kernel coefficient) parameters
- **Decision Boundary**: SVM creates optimal separating hyperplanes with maximum margin between classes
- **Support Vectors**: Only a subset of training points (support vectors) determine the final model

These metrics help you:
1. **Assess model reliability** through cross-validation
2. **Understand failure cases** through error analysis
3. **Choose appropriate metrics** for your specific problem
4. **Compare models** objectively across multiple dimensions
5. **Evaluate probability calibration** through confidence analysis
6. **Understand SVM decision-making** through support vector analysis

**Practical Implications for Tic-Tac-Toe:**
- High accuracy suggests SVM successfully learns the logical rules of tic-tac-toe winning conditions
- The model can generalize well to unseen board configurations
- Excellent performance indicates that the 9-dimensional feature space (board positions) provides sufficient information for classification
- The balanced performance across classes shows the model handles both winning and non-winning scenarios effectively
